# Domain Adaptation through Prompt Tuning

Ordinary prompting:
Write instructions using text.

Prompt tuning:
Train a small sequence of continuous vectors called a soft prompt.

The model receives:

[Learned prompt vectors] + [Input token embeddings]

During training:
- Original model parameters remain frozen.
- Only the soft prompt embeddings change.
- Gradients pass through the frozen model to train the prompt.

We will compare:
1. The original model without a soft prompt.
2. The model with an initialized soft prompt.
3. The model with a trained soft prompt.

In [29]:
%pip install -q torch "transformers>=4.45,<5" "peft>=0.13,<0.19" pandas

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [30]:
import hashlib
import math
import random

import pandas as pd
import torch

from IPython.display import display
from torch.utils.data import DataLoader

from transformers import AutoTokenizer, AutoModelForCausalLM

from peft import (
    PromptTuningConfig,
    PromptTuningInit,
    TaskType,
    get_peft_model,
)

# Choose "cpu", "cuda", or "auto".
DEVICE_CHOICE = "cpu"

if DEVICE_CHOICE == "auto":
    device = torch.device(
        "cuda" if torch.cuda.is_available() else "cpu"
    )

elif DEVICE_CHOICE == "cuda":
    if not torch.cuda.is_available():
        raise RuntimeError(
            'CUDA is unavailable. Use DEVICE_CHOICE = "cpu".'
        )
    device = torch.device("cuda")

elif DEVICE_CHOICE == "cpu":
    device = torch.device("cpu")

else:
    raise ValueError('Choose "cpu", "cuda", or "auto".')

SEED = 42
random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print("Device:", device)

if device.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))

Device: cpu


In [31]:
# Experiment settings
## The soft prompt learning rate differs from the BitFit exercise because we are optimizing a different type of parameter.

In [32]:
MODEL_NAME = "./workshop_models/distilgpt2"

NUM_VIRTUAL_TOKENS = 10

MAX_LENGTH = 96
BATCH_SIZE = 1
TRAINING_STEPS = 30
LEARNING_RATE = 1e-2

INITIAL_PROMPT_TEXT = (
    "Manufacturing maintenance vibration bearings lubrication equipment"
)

print("Virtual tokens:", NUM_VIRTUAL_TOKENS)
print("Training steps:", TRAINING_STEPS)

Virtual tokens: 10
Training steps: 30


In [33]:
## Virtual tokens are trainable embedding positions. They do not need to correspond to readable words after training.

# Training and held-out evaluation data
# Using the same data as i had used on the BitFit notebook.

In [34]:
train_texts = [
    "Predictive maintenance uses equipment condition data to plan "
    "maintenance before a failure occurs.",

    "Preventive maintenance follows a planned schedule based on "
    "time or equipment usage.",

    "Vibration monitoring helps engineers identify changes in "
    "the condition of rotating machinery.",

    "Bearing faults can produce characteristic vibration patterns. "
    "Engineers examine frequency information and operating conditions.",

    "An increase in bearing temperature may indicate lubrication "
    "problems, increased load, or other developing issues.",

    "Root mean square vibration summarizes the magnitude of a "
    "vibration signal over a measurement interval.",

    "A condition monitoring baseline describes normal equipment "
    "behavior under known operating conditions.",

    "Sensor trends should be compared with operating speed and load "
    "because these factors can influence measurements.",

    "A single abnormal reading does not confirm a machine fault. "
    "Additional measurements help support a diagnosis.",

    "Lubrication reduces friction and wear between moving surfaces. "
    "Incorrect lubrication can contribute to bearing damage.",

    "Shaft misalignment can increase vibration and mechanical stress "
    "in rotating equipment.",

    "Maintenance decisions should combine sensor measurements, "
    "inspection findings, and equipment history.",
]

eval_texts = [
    "Condition-based maintenance uses observations of machine health "
    "to decide when maintenance is needed.",

    "A technician investigating elevated vibration should consider "
    "changes in load, speed, alignment, and bearing condition.",

    "Rising bearing temperature is a symptom that requires further "
    "investigation rather than an immediate fault diagnosis.",

    "Comparing measurements with a normal operating baseline helps "
    "identify changes in equipment condition.",
]

assert set(train_texts).isdisjoint(eval_texts)

print("Training texts:", len(train_texts))
print("Evaluation texts:", len(eval_texts))

Training texts: 12
Evaluation texts: 4


In [35]:
# Loading a fresh base model and prepare batches

In [36]:
MODEL_NAME = "./workshop_models/distilgpt2"


tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    local_files_only=True,
)
tokenizer.pad_token = tokenizer.eos_token

base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    local_files_only=True,
    attn_implementation="eager",
).to(device)

base_model.config.pad_token_id = tokenizer.pad_token_id
base_model.config.use_cache = False

# tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
# tokenizer.pad_token = tokenizer.eos_token

# base_model = AutoModelForCausalLM.from_pretrained(MODEL_NAME)
# base_model.config.pad_token_id = tokenizer.pad_token_id
# base_model.config.use_cache = False

# base_model = base_model.to(device)


def collate_texts(texts):
    batch = tokenizer(
        [text + tokenizer.eos_token for text in texts],
        padding=True,
        truncation=True,
        max_length=MAX_LENGTH,
        return_tensors="pt",
    )

    labels = batch["input_ids"].clone()
    labels[batch["attention_mask"] == 0] = -100

    batch["labels"] = labels
    return batch


train_loader = DataLoader(
    train_texts,
    batch_size=BATCH_SIZE,
    shuffle=True,
    collate_fn=collate_texts,
)

eval_loader = DataLoader(
    eval_texts,
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=collate_texts,
)

print("Base model and data loaders ready.")

Base model and data loaders ready.


In [37]:
# Define evaluation functions
# For fair comparison, we score the same target positions in all three stages.
# With a soft prompt, the model can also predict the first real input token. We exclude that extra prediction here so it matches the original model’s evaluation positions.

In [38]:
@torch.no_grad()
def evaluate_loss(current_model, loader):
    current_model.eval()

    total_loss = 0.0
    total_tokens = 0

    for batch in loader:
        batch = {
            key: value.to(device)
            for key, value in batch.items()
        }

        # Pass inputs only; calculate comparable loss ourselves.
        output = current_model(
            input_ids=batch["input_ids"],
            attention_mask=batch["attention_mask"],
        )

        input_length = batch["input_ids"].shape[1]

        # PEFT prompt tuning returns logits for virtual positions too.
        # Retain only positions corresponding to real input tokens.
        real_logits = output.logits[:, -input_length:, :]

        prediction_logits = real_logits[:, :-1, :].contiguous()
        targets = batch["labels"][:, 1:].contiguous()

        loss_sum = torch.nn.functional.cross_entropy(
            prediction_logits.reshape(-1, prediction_logits.shape[-1]),
            targets.reshape(-1),
            ignore_index=-100,
            reduction="sum",
        )

        valid_tokens = (targets != -100).sum().item()

        total_loss += loss_sum.item()
        total_tokens += valid_tokens

    return total_loss / total_tokens


@torch.no_grad()
def complete_text(current_model, prompt):
    current_model.eval()

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
    ).to(device)

    output_ids = current_model.generate(
        **inputs,
        max_new_tokens=45,
        do_sample=False,
        pad_token_id=tokenizer.pad_token_id,
        eos_token_id=tokenizer.eos_token_id,
        use_cache=True,
    )

    prompt_length = inputs["input_ids"].shape[1]

    return tokenizer.decode(
        output_ids[0, prompt_length:],
        skip_special_tokens=True,
    ).strip()


test_prompts = [
    "Predictive maintenance is",
    "When bearing temperature increases,",
    "Vibration monitoring helps engineers",
]

In [39]:
# Evaluate the original model

In [40]:
original_loss = evaluate_loss(base_model, eval_loader)

original_completions = {
    prompt: complete_text(base_model, prompt)
    for prompt in test_prompts
}

print(f"Original model loss: {original_loss:.4f}")
print(f"Original model perplexity: {math.exp(original_loss):.2f}")

for prompt, completion in original_completions.items():
    print("\nPROMPT:", prompt)
    print("CONTINUATION:", completion)

Original model loss: 5.4397
Original model perplexity: 230.38

PROMPT: Predictive maintenance is
CONTINUATION: a key component of the software. It is a key component of the software. It is a key component of the software. It is a key component of the software. It is a key component of the software. It is

PROMPT: When bearing temperature increases,
CONTINUATION: the temperature increases, and the temperature decreases.

PROMPT: Vibration monitoring helps engineers
CONTINUATION: to understand the impact of the stress on the brain.


In [41]:
# Attach the trainable soft prompt
# We initialize the prompt using embeddings of domain-related text. This initialization is only the starting point; training subsequently updates the vectors

In [42]:
# prompt_config = PromptTuningConfig(
#     task_type=TaskType.CAUSAL_LM,
#     num_virtual_tokens=NUM_VIRTUAL_TOKENS,
#     prompt_tuning_init=PromptTuningInit.TEXT,
#     prompt_tuning_init_text=INITIAL_PROMPT_TEXT,
#     tokenizer_name_or_path=MODEL_NAME,
# )

# model = get_peft_model(base_model, prompt_config)
# model = model.to(device)

# model.print_trainable_parameters()

prompt_config = PromptTuningConfig(
    task_type=TaskType.CAUSAL_LM,
    num_virtual_tokens=NUM_VIRTUAL_TOKENS,
    prompt_tuning_init=PromptTuningInit.TEXT,
    prompt_tuning_init_text=INITIAL_PROMPT_TEXT,
    tokenizer_name_or_path=MODEL_NAME,
    tokenizer_kwargs={"local_files_only": True},
)

model = get_peft_model(base_model, prompt_config)
model = model.to(device)

model.print_trainable_parameters()

trainable_names = [
    name
    for name, parameter in model.named_parameters()
    if parameter.requires_grad
]

print("\nTrainable parameter names:")
for name in trainable_names:
    print(name)

assert all(
    not parameter.requires_grad
    for parameter in model.base_model.parameters()
)

prompt_before = (
    model.prompt_encoder["default"]
    .embedding.weight.detach().cpu().clone()
)

print("\nSoft prompt shape:", tuple(prompt_before.shape))
print("Expected shape: virtual tokens × embedding dimension")

trainable params: 7,680 || all params: 81,920,256 || trainable%: 0.0094

Trainable parameter names:
prompt_encoder.default.embedding.weight

Soft prompt shape: (10, 768)
Expected shape: virtual tokens × embedding dimension


In [43]:
### For DistilGPT2, 10 vectors of dimension 768 give 7,680 trainable parameters.

In [44]:
# Recording the initialized prompt baseline

#### This separates the effect of adding a prompt from the effect of training it.

In [45]:
def frozen_parameter_fingerprints(current_model):
    return {
        name: hashlib.sha256(
            parameter.detach().cpu().contiguous().numpy().tobytes()
        ).hexdigest()
        for name, parameter in current_model.named_parameters()
        if not parameter.requires_grad
    }


frozen_before = frozen_parameter_fingerprints(model)

initialized_loss = evaluate_loss(model, eval_loader)

initialized_completions = {
    prompt: complete_text(model, prompt)
    for prompt in test_prompts
}

print(f"Initialized soft prompt loss: {initialized_loss:.4f}")

C:\Users\fedex\anaconda3\envs\ABMLLM\Lib\site-packages\peft\peft_model.py:2141: UserWarning: Position ids are not supported for parameter efficient tuning. Ignoring position ids.
  warnings.warn("Position ids are not supported for parameter efficient tuning. Ignoring position ids.")


Initialized soft prompt loss: 5.0060


In [46]:
# Train only the soft prompt

In [47]:
trainable_parameters = [
    parameter
    for parameter in model.parameters()
    if parameter.requires_grad
]

optimizer = torch.optim.AdamW(
    trainable_parameters,
    lr=LEARNING_RATE,
    weight_decay=0.0,
)

training_history = []
train_iterator = iter(train_loader)

model.train()

for step in range(1, TRAINING_STEPS + 1):
    try:
        batch = next(train_iterator)
    except StopIteration:
        train_iterator = iter(train_loader)
        batch = next(train_iterator)

    batch = {
        key: value.to(device)
        for key, value in batch.items()
    }

    optimizer.zero_grad(set_to_none=True)

    output = model(**batch)
    loss = output.loss

    loss.backward()

    torch.nn.utils.clip_grad_norm_(
        trainable_parameters,
        max_norm=1.0,
    )

    optimizer.step()

    training_history.append({
        "step": step,
        "training_loss": loss.item(),
    })

    if step == 1 or step % 5 == 0 or step == TRAINING_STEPS:
        print(
            f"Step {step:3d}/{TRAINING_STEPS} "
            f"| Training loss: {loss.item():.4f}"
        )

print("\nPrompt tuning complete.")

Step   1/30 | Training loss: 5.7656
Step   5/30 | Training loss: 4.7257
Step  10/30 | Training loss: 4.9279
Step  15/30 | Training loss: 4.9804
Step  20/30 | Training loss: 4.9959
Step  25/30 | Training loss: 4.8943
Step  30/30 | Training loss: 4.6209

Prompt tuning complete.


In [48]:
### PEFT handles adding the virtual embeddings and masking their target positions during training.

# Compare evaluation results

In [49]:
trained_loss = evaluate_loss(model, eval_loader)

results_df = pd.DataFrame([
    {
        "stage": "Original model — no soft prompt",
        "evaluation_loss": original_loss,
        "perplexity": math.exp(original_loss),
    },
    {
        "stage": "Initialized soft prompt",
        "evaluation_loss": initialized_loss,
        "perplexity": math.exp(initialized_loss),
    },
    {
        "stage": "Trained soft prompt",
        "evaluation_loss": trained_loss,
        "perplexity": math.exp(trained_loss),
    },
])

display(results_df)

print(
    "Loss reduction from prompt training:",
    f"{initialized_loss - trained_loss:.4f}",
)

print(
    "Loss reduction versus original model:",
    f"{original_loss - trained_loss:.4f}",
)

,stage,evaluation_loss,perplexity
0,Original model — no soft prompt,5.439722,230.378014
1,Initialized soft prompt,5.006021,149.309491
2,Trained soft prompt,4.185585,65.731937


Loss reduction from prompt training: 0.8204
Loss reduction versus original model: 1.2541


In [50]:
### Positive reduction means improvement. Training may improve the initialized prompt without outperforming the original model, so inspect both comparisons.

# Compare completions

In [51]:
trained_completions = {
    prompt: complete_text(model, prompt)
    for prompt in test_prompts
}

for prompt in test_prompts:
    print("\nPROMPT:", prompt)

    print("\nORIGINAL MODEL:")
    print(original_completions[prompt])

    print("\nINITIALIZED SOFT PROMPT:")
    print(initialized_completions[prompt])

    print("\nTRAINED SOFT PROMPT:")
    print(trained_completions[prompt])

    print("=" * 70)

C:\Users\fedex\anaconda3\envs\ABMLLM\Lib\site-packages\peft\peft_model.py:2141: UserWarning: Position ids are not supported for parameter efficient tuning. Ignoring position ids.
  warnings.warn("Position ids are not supported for parameter efficient tuning. Ignoring position ids.")



PROMPT: Predictive maintenance is

ORIGINAL MODEL:
a key component of the software. It is a key component of the software. It is a key component of the software. It is a key component of the software. It is a key component of the software. It is

INITIALIZED SOFT PROMPT:
a process of testing and testing the bearings and bearings for the manufacturing of bearings and bearings for the manufacturing of bearings and bearings for the manufacturing of bearings and bearings for the manufacturing of bearings and bearings for the manufacturing of bearings and bearings

TRAINED SOFT PROMPT:
a critical component of the maintenance of the system.

PROMPT: When bearing temperature increases,

ORIGINAL MODEL:
the temperature increases, and the temperature decreases.

INITIALIZED SOFT PROMPT:
the bearings will be replaced with a new bearings. The bearings will be replaced with a new bearings. The bearings will be replaced with a new bearings. The bearings will be replaced with a new bearings. The be

In [52]:
#### A short run on 12 sentences demonstrates the mechanism. It does not guarantee accurate maintenance advice or fluent answers.

In [53]:
# Just Verifying What CChanged

In [54]:
prompt_after = (
    model.prompt_encoder["default"]
    .embedding.weight.detach().cpu().clone()
)

prompt_change = prompt_after - prompt_before

print(
    "Mean absolute prompt change:",
    prompt_change.abs().mean().item(),
)

print(
    "Largest absolute prompt change:",
    prompt_change.abs().max().item(),
)

frozen_after = frozen_parameter_fingerprints(model)

changed_frozen = [
    name
    for name in frozen_before
    if frozen_before[name] != frozen_after[name]
]

print("Changed frozen parameter tensors:", len(changed_frozen))

assert not torch.equal(prompt_before, prompt_after)
assert len(changed_frozen) == 0

print("\nVerified: the soft prompt changed; original weights stayed identical.")

display(pd.DataFrame({
    "virtual_token": range(NUM_VIRTUAL_TOKENS),
    "mean_absolute_change": (
        prompt_change.abs().mean(dim=1).numpy()
    ),
}))

Mean absolute prompt change: 0.06060114875435829
Largest absolute prompt change: 0.22574353218078613
Changed frozen parameter tensors: 0

Verified: the soft prompt changed; original weights stayed identical.


,virtual_token,mean_absolute_change
0,0,0.064388
1,1,0.059631
2,2,0.054213
3,3,0.053617
4,4,0.060210
5,5,0.057218
6,6,0.060125
7,7,0.062128
8,8,0.066369
9,9,0.068113


## Participant exercises

1. Change NUM_VIRTUAL_TOKENS from 10 to 20.
   How does the trainable parameter count change?

2. Compare 30 and 60 training steps.

3. Try LEARNING_RATE = 1e-3 instead of 1e-2.

4. Compare the initialized prompt with the trained prompt.
   Why do we need both measurements?

5. Does better held-out loss guarantee factual correctness?

For a fresh experiment:
Change settings in Cell 4, then rerun Cells 6–14.

## BitFit versus prompt tuning

BitFit:
Updates existing bias parameters inside the model.

Prompt tuning:
Adds and trains input-level prompt embeddings.
All original model parameters remain frozen.

## Main takeaway

The learned prompt is a matrix of numerical vectors.
It is not a readable instruction that we can copy into Ollama.